# 03 - Prepare Data

Name: Usha Oudeacoumar
Student ID: 20057009
Email: 20057009@mydbs.ie

This notebook builds the clean dataset we will actually use for modelling. For every
year it loads the raw data, keeps only adults, builds the admit column (our target),
keeps only the feature columns we picked in notebook 01, and fixes one known problem
with the IMMEDR column (triage level) where 2011 used a different missing code than
later years. Then it stacks all years together and saves one big csv file in
data/processed that the rest of the notebooks will use.

In [1]:
import sys
sys.path.append(".")
import os
import pandas as pd
from utils import STUDY_YEARS, PROC_DIR, load_year_raw, add_admit_flag, filter_adults, load_json

os.makedirs(PROC_DIR, exist_ok=True)

cfg = load_json("../configs/study_config.json")
feature_cols = cfg["feature_cols"]
feature_cols

['AGE',
 'SEX',
 'ETHIM',
 'RACER',
 'TEMPF',
 'PULSE',
 'RESPR',
 'BPSYS',
 'BPDIAS',
 'POPCT',
 'ARREMS',
 'IMMEDR',
 'PAINSCALE',
 'RESIDNCE',
 'RFV1']

In [2]:
# 2011 did not have -8/-9 codes for IMMEDR (triage level), it used code 07 to mean
# unknown/not recorded instead. make it match 2012+ so the column means the same
# thing across all years
def fix_immedr_2011(df, yr):
    if yr == 2011 and "IMMEDR" in df.columns:
        df = df.copy()
        df.loc[df["IMMEDR"] == "07", "IMMEDR"] = "-9"
    return df

In [3]:
# do the whole clean up steps for one year and return just the columns we need
def clean_one_year(yr):
    df = load_year_raw(yr)
    df = filter_adults(df)
    df = add_admit_flag(df)
    df = fix_immedr_2011(df, yr)

    keep_cols = ["yr", "admit"] + [c for c in feature_cols if c in df.columns]
    df = df[keep_cols]

    # some feature columns don't exist in every year (eg PAINSCALE missing in 2011)
    # add them back in as blank so every year has the same columns
    for c in feature_cols:
        if c not in df.columns:
            df[c] = pd.NA

    return df

In [4]:
# run it for every year and stack them all into one dataframe
all_years_list = []
for yr in STUDY_YEARS:
    print("cleaning", yr)
    df_yr = clean_one_year(yr)
    all_years_list.append(df_yr)

clean_df = pd.concat(all_years_list, ignore_index=True)
print(clean_df.shape)
clean_df.head()

cleaning 2011
cleaning 2012
cleaning 2013
cleaning 2014
cleaning 2015
cleaning 2016
cleaning 2017
cleaning 2018
cleaning 2019
(160526, 17)


,yr,admit,AGE,SEX,ETHIM,RACER,TEMPF,PULSE,RESPR,BPSYS,BPDIAS,POPCT,ARREMS,IMMEDR,RESIDNCE,RFV1,PAINSCALE
0,2011,0,033,1,2,1,0972,067,016,105,072,-09,02,02,01,15451,<NA>
1,2011,1,053,2,2,1,0981,080,018,183,120,100,02,04,01,12300,<NA>
2,2011,0,052,2,2,1,0996,046,016,120,080,098,-9,03,-9,10150,<NA>
3,2011,0,032,1,2,1,0989,091,016,133,083,-09,02,01,01,16500,<NA>
4,2011,1,046,1,2,1,0978,075,018,222,118,098,02,04,01,10501,<NA>


In [5]:
# quick sanity check, admit rate per year should look the same as notebook 02
clean_df.groupby("yr")["admit"].agg(["count", "mean"])

,count,mean
yr,,
2011,24466,0.149105
2012,23275,0.131429
2013,19403,0.119930
2014,18228,0.108789
2015,16393,0.105167
2016,15242,0.102086
2017,12793,0.118737
2018,15827,0.137550
2019,14899,0.127257


In [6]:
# save the clean dataset so the next notebooks don't have to redo all this work
out_path = os.path.join(PROC_DIR, "clean_data.csv")
clean_df.to_csv(out_path, index=False)
print("saved", out_path)

saved ../data/processed/clean_data.csv
